# Investigate the original Omniglot ResNet18 data

**Stage 0 — data exploration (no model training).**

This notebook reads the official researchers' `.h5` file from the pinned `external/icl-dynamics` submodule. You can change the parameters in the next cell and rerun to inspect any class or exemplar. 

The dataset contains **learned feature vectors, not raw handwriting images**. There is no ground-truth character name attached to each shuffled class ID. The plots below are exploratory; they do not establish induction-head mechanisms or model collapse.

In [ ]:
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt

# Change these settings and rerun all cells.
SEED = 42
CLASS_ID = 0
EXEMPLAR_ID = 0
PLOT_CLASSES = 24
DATASET_KEY = "resnet18/224/feat"

roots = [Path.cwd(), *Path.cwd().parents]
candidates = []
for root in roots:
    candidates.extend([
        root / "external" / "icl-dynamics" / "omniglot_resnet18_randomized_order_s0.h5",
        root / "data" / "raw" / "omniglot_resnet18_randomized_order_s0.h5",
    ])
DATA_PATH = next((p for p in candidates if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Official HDF5 missing. Run 'git submodule update --init --recursive' "
        "at the repository root, then restart the notebook."
    )
print("Dataset:", DATA_PATH.resolve())

## 1. What's actually inside this file?

We inspect the HDF5 groups and datasets rather than assuming their contents. The original code expects `resnet18/224/feat`. The three axes mean **class, exemplar, feature coordinate**.

In [ ]:
with h5py.File(DATA_PATH, "r") as f:
    def describe(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"{name:<35} shape={obj.shape}, dtype={obj.dtype}")
    f.visititems(describe)
    source_shape = f[DATASET_KEY].shape
    # Reserve 100 final test classes; inspect only training + validation rows.
    features = np.array(f[DATASET_KEY][:1523])
print("Full published HDF5 dimensions:", source_shape)
print("Development subset (train + validation only):", features.shape)
print("Final 100 class IDs (1523..1622) are NOT loaded or scored here.")
print("Contains NaN or Inf:", not np.isfinite(features).all())
print("Size in memory (MiB):", round(features.nbytes / 2**20, 2))
N_CLASSES, N_EXEMPLARS, N_FEATURES = features.shape

## 2. What does one exemplar look like numerically?

A line plot of its vector coordinates is **not** a handwriting picture. It tells us the dimensionality, magnitudes and sparsity of the *representation* a downstream transformer receives.

In [ ]:
assert 0 <= CLASS_ID < N_CLASSES and 0 <= EXEMPLAR_ID < N_EXEMPLARS
vector = features[CLASS_ID, EXEMPLAR_ID]
print(f"Class {CLASS_ID}, exemplar {EXEMPLAR_ID}")
print("First 20 feature coordinates:", np.round(vector[:20], 3))
print("L2 norm:", float(np.linalg.norm(vector)))
fig, ax = plt.subplots(figsize=(10.5, 3.4))
ax.plot(vector, linewidth=1)
ax.set(title=f"Precomputed embedding, class {CLASS_ID} exemplar {EXEMPLAR_ID}",
       xlabel="Feature coordinate", ylabel="Activation")
fig.tight_layout()
plt.show()

## 3. Do exemplars from the same class resemble one another?

We compare two types of cosine similarity:
- **Within class:** two different drawings of the same character.
- **Between classes:** drawings from distinct classes.

Large overlap in these histograms would show that a simple cosine similarity is imperfect. It does **not** automatically imply the transformer cannot learn matching.

In [ ]:
def cosine(a, b):
    return np.sum(a * b, axis=-1) / np.maximum(
        np.linalg.norm(a, axis=-1) * np.linalg.norm(b, axis=-1), 1e-12)

rng = np.random.default_rng(SEED)
selected = rng.choice(N_CLASSES, size=min(N_CLASSES, 800), replace=False)
within = cosine(features[selected, 0], features[selected, 1])
between = cosine(features[selected, 0], features[(selected + 1) % N_CLASSES, 0])
print("Mean within-class cosine:", round(float(within.mean()), 4))
print("Mean between-class cosine:", round(float(between.mean()), 4))
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(within, bins=35, alpha=.65, label="Same class")
ax.hist(between, bins=35, alpha=.65, label="Different class")
ax.set(title="Embedding similarity by class relationship", xlabel="Cosine similarity", ylabel="Number of comparisons")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Are the five exemplars of a selected class mutually similar?

Change `CLASS_ID` to investigate any class. Remember that these class IDs were randomised.

In [ ]:
X = features[CLASS_ID]
similarity = cosine(X[:, None, :], X[None, :, :])
fig, ax = plt.subplots(figsize=(5.5, 4.5))
hm = ax.imshow(similarity, vmin=-1, vmax=1, cmap="viridis")
ax.set(title=f"Class {CLASS_ID}: exemplar similarities",
       xlabel="Exemplar index", ylabel="Exemplar index",
       xticks=range(N_EXEMPLARS), yticks=range(N_EXEMPLARS))
fig.colorbar(hm, ax=ax, label="Cosine similarity")
fig.tight_layout()
plt.show()

## 5. Project feature vectors to two dimensions with PCA

PCA is an exploratory way to see which class exemplars cluster together. Two dimensions cannot preserve all relationships in high-dimensional feature space, and plotting the first few classes does not establish global separability.

In [ ]:
rng = np.random.default_rng(SEED)
classes_to_plot = rng.choice(N_CLASSES, size=min(max(2, PLOT_CLASSES), N_CLASSES), replace=False)
cloud = features[classes_to_plot].reshape(-1, N_FEATURES).astype(np.float64)
cloud = cloud - cloud.mean(axis=0)
_, _, directions = np.linalg.svd(cloud, full_matrices=False)
coords = cloud @ directions[:2].T
fig, ax = plt.subplots(figsize=(8, 5.5))
im = ax.scatter(coords[:,0], coords[:,1],
                c=np.repeat(np.arange(len(classes_to_plot)), N_EXEMPLARS),
                cmap="tab20", s=24, alpha=.8)
fig.colorbar(im, ax=ax, label="Sampled class index")
ax.set(title=f"PCA of {len(classes_to_plot)} sampled character classes",
       xlabel="PC 1", ylabel="PC 2")
fig.tight_layout()
plt.show()

## 6. Why does the training data need random labels?

The actual task is not classifying a character's permanent label. **Each episode independently assigns numeric labels to the chosen classes.** The query must copy the numeric label attached to the matching character in *that* episode.

The example below is a human-readable sketch. The upstream code uses the feature vectors rather than literal class-name tokens.

In [ ]:
rng = np.random.default_rng(SEED)
a, b = rng.choice(50, size=2, replace=False)
label_a, label_b = rng.choice(5, size=2, replace=False)
query = int(rng.choice([a,b]))
oracle = int(label_a if query == a else label_b)
print(f"Episode 1: C{a} {label_a} | C{b} {label_b} | query C{query} -> {oracle}")
print(f"Label-swap: C{a} {label_b} | C{b} {label_a} | query C{query}"
      f" -> {int(label_b if query==a else label_a)}")
print("A memorised global class-to-label mapping cannot solve both correctly.")

## 7. Questions to answer before training models

1. How separable are same-class versus different-class ResNet18 exemplars?
2. Why can a model still need in-context induction when class features are informative?
3. Which held-out tests isolate **matching**, and which isolate **copying**?
4. Does your proposed next-symbol generator sample from the **eligible symbols in context**, or from a global vocabulary?
5. If we later observe generation-to-generation degradation, what matched baseline would rule out ordinary sampling drift?
6. What does the old *symbolic* archive establish, and which causal link has it **not** established?

**Do not use previously opened final-test results to choose new hypotheses, heads or hyperparameters for a supposedly fresh confirmatory test.** See `docs/OPTION_B_PLAN.md`.

## 7. Our train / validation / test split (source-matched)

**A dataset does not have just one split.** Singh et al. divide the *character identities*, *drawings of each character*, and *random two-label mappings* separately.

| Split dimension | Train | Validation | Test |
| --- | --- | --- | --- |
| Character classes | 50 (indices 0–49) | 1,473 (50–1522) | 100 (1523–1622) |
| Exemplars per class | one (ID 0) | four (IDs 1–4) | none extra |
| Unordered numeric label pairs | eight | two | none extra |

The **original paper's test-class evaluator is periodic**, not a truly untouched final test. For this course project, our protocol reserves the 100 test classes for a once-only final *model* evaluation, not model selection. The source-verified details are recorded in `docs/TRAIN_VAL_TEST_PROTOCOL.md` and `docs/data-audit/reference_split_manifest.json`.

## 8. Figure 2 — why the overlapping coloured histograms matter

For any query handwritten character:
- **Same-class** cosine measures how closely a different drawing resembles the *correct* reference.
- **Different-class** cosine measures how closely it resembles a *wrong* reference (a distractor).

If the first distribution were always to the right of the second, character identity would be easy to recover from these embeddings. They overlap, however, so *some wrong characters look more similar numerically than the correct one*.

**Important:** the two histograms are input-data measurements. They do **not** show whether the transformer learned an induction head. The plot below is a newer version that distinguishes training classes from validation classes and does **not** score final-test classes.

![Figure 2: training and validation similarities](../docs/data-audit/06_split_similarities.png)

## 9. Figure 5 — one wrong answer versus hundreds

With **two** candidate characters the correct one competes with just **one** wrong reference. With **50** candidates it competes with 49 wrong references. With **1,473** it competes with 1,472.

A query might score 0.90 against its correct character and 0.81 against most others, yet one unusually similar *wrong* character could score 0.93 and beat it.

This is why the classification becomes harder when the candidate set grows. We computed the **expected win probability** over all equally likely distractor subsets instead of relying on arbitrary numbers of random draws.

This still measures a simple cosine rule, **not** a trained transformer predicting in-context labels.

![Figure 5: matching difficulty rises with candidate-set size](../docs/data-audit/07_matching_vs_candidates.png)

In [ ]:
import json
from pathlib import Path

def find_report_file(filename):
    for root in (Path.cwd(), *Path.cwd().parents):
        p = root / "docs" / "data-audit" / filename
        if p.is_file():
            return p
    raise FileNotFoundError(
        "Run the GitHub data-audit workflow or clone the repository with "
        "its generated docs/data-audit folder. Missing " + filename
    )

split = json.loads(find_report_file("reference_split_manifest.json").read_text())
diagnostics = json.loads(find_report_file("split_diagnostics.json").read_text())
print("Eight training label pairs:", split["label_pairs"]["train"])
print("Two held-out validation pairs:", split["label_pairs"]["val"])
print("Final test classes scored by these diagnostics:",
      diagnostics["test_class_indices_ever_scored"])
for name in ("train", "val"):
    d = diagnostics["partitions"][name]
    print(f"{name.title()} ({d['num_classes']} classes): "
          f"same mean cosine={d['mean_same_class_cosine']:.3f}, "
          f"wrong mean cosine={d['mean_random_different_class_cosine']:.3f}, "
          f"2-way accuracy={100*d['exact_expected_two_choice_accuracy']:.2f}%, "
          f"all-{d['num_classes']} accuracy={100*d['all_partition_class_top1_accuracy']:.2f}%")

## 10. The next scientific step

The diagrams above tell us whether **input feature vectors** contain enough information to compare characters. A real model-level study will require trained checkpoint weights.

After implementation, this *same notebook* can display:

1. **Attention maps** identifying which context position matches the query.
2. **Knockout/retain-only experiments** to see whether individual heads causally contribute to contextual prediction.
3. **Forced-attention matching controls** to separate *finding* a label from *copying* it.
4. **Data repair versus unmodified synthetic training**, with carefully matched training seeds and test protocols.

Data-repair code is located in `src/target_repairs.py`. Its unit-test demonstrations are **not** new evidence of collapse. Training and circuit tests still need to be implemented and reproduced.

## Next stage — interpreting a trained model (after approval)

These feature arrays are inputs to a neural network; **they contain no attention heads that can be ablated**.

Once we have validated model checkpoints, this same notebook can load quantitative results and interactive figures from well-tested experiment modules:

- **Observation:** attention from the query to the right/wrong contextual label; previous-token attention strength.
- **Causal ablation:** disable a head's output and measure the change in held-out accuracy/log loss.
- **Causal matching repair:** replace selected query-position attention patterns with oracle matches and measure recovery.
- **Data intervention:** repair selected synthetic targets *before* retraining, then compare matched models.

Interpret the last two separately: repairing a model's attention at test time does not establish what training-data defect produced the failure. See `docs/OPTION_B_PLAN.md` and `docs/NEXT_STAGE_PLAN.md`.

No model has been trained or ablated in this repository yet. Notebook 1 will remain the **single interactive notebook**; implementation code can live in reusable modules instead of notebook-only cells.